# Fine-tuning and Evaluating `facebook/opt-125m` for News Simplification

This notebook:

1. Loads `data/processed/train.jsonl` and `data/processed/validation.jsonl`
2. Fine-tunes `facebook/opt-125m` to continue the fixed prompt template
   with the reference (gold) summary
3. Saves the fine-tuned model
4. Generates structured output for every article in `data/processed/test.jsonl`
5. Evaluates the generated output with:
   - **ROUGE-1, ROUGE-2, ROUGE-L**
   - **Flesch Reading Ease**
   - **Compression Ratio**
   - **Generation Time**

**Note on supervision:** XSum only provides a single gold *summary* per
article — there is no gold "key points / vocabulary / quiz" text. Fine-tuning
therefore trains the model to continue the fixed prompt with the gold
summary (the part of the response we can supervise directly), while the
prompt template itself still asks the model to produce all four sections
at generation/inference time.


## 0. Setup

In [1]:
import os
import re
import json
import time

import pandas as pd
import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
)
from rouge_score import rouge_scorer
import textstat

PROCESSED_DIR = "data/processed"
TRAIN_PATH = os.path.join(PROCESSED_DIR, "train.jsonl")
VAL_PATH = os.path.join(PROCESSED_DIR, "validation.jsonl")
TEST_PATH = os.path.join(PROCESSED_DIR, "test.jsonl")

MODEL_NAME = "facebook/opt-125m"
FINETUNED_MODEL_DIR = "models/opt-125m-news-simplifier"
RESULTS_DIR = "results"
os.makedirs(FINETUNED_MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

MAX_TRAIN_LENGTH = 1024          # max tokens per training example (prompt + summary)
MAX_CONTEXT = 2048               # OPT context window
MAX_NEW_TOKENS = 200             # max tokens to generate at inference time

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


Using device: cuda


## 1. Load Train / Validation / Test Splits

Reads the JSONL splits produced by `data_split.ipynb`.


In [2]:
def load_jsonl(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Could not find '{path}'. Run the preprocessing and data_split "
            "notebooks first to generate this file."
        )
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return pd.DataFrame(records)


train_df = load_jsonl(TRAIN_PATH)
val_df = load_jsonl(VAL_PATH)
test_df = load_jsonl(TEST_PATH)

print(f"Train:      {len(train_df)} rows")
print(f"Validation: {len(val_df)} rows")
print(f"Test:       {len(test_df)} rows")
train_df.head()


Train:      800 rows
Validation: 100 rows
Test:       100 rows


,article_id,article,summary,chunk,chunk_count,prompt,prompt_word_count,prompt_token_length,input_ids
0,12621383,Tanweer was a sports science graduate whose in...,Shehzad Tanweer was born in Bradford but lived...,[Tanweer was a sports science graduate whose i...,1,[You are a news simplification assistant. Simp...,[596],[758],"[[2, 1185, 32, 10, 340, 24988, 5000, 3167, 4, ..."
1,37341636,Media playback is not supported on this device...,Wales outgunned Israel as they coasted to a co...,[Media playback is not supported on this devic...,1,[You are a news simplification assistant. Simp...,[389],[513],"[[2, 1185, 32, 10, 340, 24988, 5000, 3167, 4, ..."
2,37895403,South West Trains services were delayed by the...,Rail services between Berkshire and London hav...,[South West Trains services were delayed by th...,1,[You are a news simplification assistant. Simp...,[234],[338],"[[2, 1185, 32, 10, 340, 24988, 5000, 3167, 4, ..."
3,39168180,Copthorne Barracks in Shrewsbury was one almos...,Ex servicemen and women should have priority t...,[Copthorne Barracks in Shrewsbury was one almo...,1,[You are a news simplification assistant. Simp...,[376],[503],"[[2, 1185, 32, 10, 340, 24988, 5000, 3167, 4, ..."
4,12640700,A court official confirmed that the papers wer...,Lawyers for Wikileaks founder Julian Assange h...,[A court official confirmed that the papers we...,1,[You are a news simplification assistant. Simp...,[257],[335],"[[2, 1185, 32, 10, 340, 24988, 5000, 3167, 4, ..."


## 2. Prepare the Fine-tuning Dataset

Each training example is built by taking the fixed **prompt** (article
inserted into the template, ending in `Response:\nSummary:\n`) and
appending the **gold XSum summary** as the target continuation.

The loss is only computed on the summary tokens: prompt tokens are masked
with `-100` in the `labels` so the model is not penalised for "predicting"
the article/instructions it was given, only for the text it must generate.


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# OPT's tokenizer has no pad token by default; reuse eos_token for padding.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


TRAIN_PROMPT_TEMPLATE = """You are a news simplification assistant. Read the following news article and write a short, simplified summary using clear and simple English. Preserve all important names, dates, locations, and numerical information. Do not add or infer information that is not explicitly stated in the article.

Article:
{article}

Response:
Summary:
"""


class NewsSimplificationDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int = MAX_TRAIN_LENGTH, prompt_template=None):
        self.prompts = df["prompt"].tolist()
        self.articles = df["article"].tolist() if "article" in df.columns else [None] * len(df)
        self.summaries = df["summary"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.prompt_template = prompt_template

    def __len__(self):
        return len(self.prompts)

    def build_prompt(self, idx):
        prompt = self.prompts[idx]
        if isinstance(prompt, list):
            if self.prompt_template is None:
                raise ValueError("Training prompt template is required for dataset rows with chunked prompts.")
            article = self.articles[idx]
            return self.prompt_template.format(article=article)
        return prompt

    def __getitem__(self, idx):
        prompt = self.build_prompt(idx)
        summary = self.summaries[idx].strip()

        prompt_ids = self.tokenizer(prompt, truncation=True,
                                     max_length=self.max_length)["input_ids"]

        target_ids = self.tokenizer(
            summary + self.tokenizer.eos_token,
            truncation=True,
            max_length=max(1, self.max_length - len(prompt_ids)),
            add_special_tokens=False,
        )["input_ids"]

        input_ids = prompt_ids + target_ids
        input_ids = input_ids[: self.max_length]

        labels = [-100] * len(prompt_ids) + target_ids
        labels = labels[: self.max_length]

        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels,
        }
    
def collate_fn(batch, pad_token_id: int):
    max_len = max(len(ex["input_ids"]) for ex in batch)

    input_ids, attention_mask, labels = [], [], []
    for ex in batch:
        pad_len = max_len - len(ex["input_ids"])
        input_ids.append(ex["input_ids"] + [pad_token_id] * pad_len)
        attention_mask.append(ex["attention_mask"] + [0] * pad_len)
        labels.append(ex["labels"] + [-100] * pad_len)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

train_dataset = NewsSimplificationDataset(
    train_df,
    tokenizer,
    prompt_template=TRAIN_PROMPT_TEMPLATE,
)

val_dataset = NewsSimplificationDataset(
    val_df,
    tokenizer,
    prompt_template=TRAIN_PROMPT_TEMPLATE,
)

print(f"Prepared {len(train_dataset)} training examples and {len(val_dataset)} validation examples")

Prepared 800 training examples and 100 validation examples


## 3. Load `facebook/opt-125m` and Fine-tune

`Trainer` handles the training loop, periodic validation, and checkpointing.
Batch size / epochs are kept small since `opt-125m` and this dataset are
both modest in size — increase them if you have a GPU with more memory.


In [4]:
print("\nConverting model to float32")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))
model = model.float()
model.to(DEVICE)
print(f"Model dtype: {next(model.parameters()).dtype}")
print(f"Model device: {next(model.parameters()).device}")


Converting model to float32


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Model dtype: torch.float32
Model device: cuda:0


In [5]:
from functools import partial

training_args = TrainingArguments(
    output_dir=os.path.join(FINETUNED_MODEL_DIR, "checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2, # Effective batch size = 8
    learning_rate=5e-5,            # REDUCED from 1e-4 for stable fine-tuning
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,                     
    dataloader_num_workers=0,      
    remove_unused_columns=False,
    report_to=[],
)

# Use partial instead of lambda to make it multiprocessing-safe
safe_collator = partial(collate_fn, pad_token_id=tokenizer.pad_token_id)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=safe_collator,
)

train_result = trainer.train()
print(train_result)

Epoch,Training Loss,Validation Loss
1,2.468511,2.322170
2,1.531084,2.346534
3,1.092863,2.418116


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=300, training_loss=1.689399522145589, metrics={'train_runtime': 120.5247, 'train_samples_per_second': 19.913, 'train_steps_per_second': 2.489, 'total_flos': 691333807104000.0, 'train_loss': 1.689399522145589, 'epoch': 3.0})


## 4. Save the Fine-tuned Model

Saves the fine-tuned weights and tokenizer to `models/opt-125m-news-simplifier`
so they can be reloaded later without retraining.


In [6]:
trainer.save_model(FINETUNED_MODEL_DIR)
tokenizer.save_pretrained(FINETUNED_MODEL_DIR)
print(f"Fine-tuned model saved to '{FINETUNED_MODEL_DIR}'")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved to 'models/opt-125m-news-simplifier'


## 5. Generate Structured Output on the Test Set

For every test article, the fine-tuned model generates a continuation of
the fixed prompt (Summary / Key Points / Vocabulary / Quiz). Generation
time is measured per example for the **Generation Time** metric.


In [ ]:
prompt_templates_path = os.path.join(PROCESSED_DIR, "prompt_templates.json")
with open(prompt_templates_path, "r", encoding="utf-8") as f:
    prompt_templates = json.load(f)

chunk_prompt_template = prompt_templates["chunk_prompt_template"]
final_prompt_template = prompt_templates["final_prompt_template"]

finetuned_model = AutoModelForCausalLM.from_pretrained(FINETUNED_MODEL_DIR).to(DEVICE)
finetuned_model.eval()


def generate_response(prompt: str, max_new_tokens: int = MAX_NEW_TOKENS):
    inputs = tokenizer(prompt, truncation=True, max_length=MAX_CONTEXT,
                        return_tensors="pt").to(DEVICE)
    input_len = inputs["input_ids"].shape[1]

    start = time.time()
    with torch.no_grad():
        output_ids = finetuned_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
        )
    elapsed = time.time() - start

    generated_ids = output_ids[0][input_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return generated_text.strip(), elapsed


def generate_chunk_summary(chunk_prompt: str, max_new_tokens: int = 100):
    return generate_response(chunk_prompt, max_new_tokens=max_new_tokens)


def build_final_prompt(chunk_summaries):
    return final_prompt_template.format(chunk_summaries="\n\n".join(chunk_summaries))


def is_chunked_prompt(prompt):
    return isinstance(prompt, list)


generated_texts = []
generation_times = []

for i, row in test_df.iterrows():
    prompts = row["prompt"] if is_chunked_prompt(row["prompt"]) else [row["prompt"]]
    
    total_time = 0.0
    
    # If it's a single prompt, skip Map-Reduce and just summarize once
    if len(prompts) == 1:
        gen_text, gen_time = generate_response(prompts[0])
        total_time += gen_time
        
    else:
        # Multiple chunks
        chunk_summaries = []
        for chunk_prompt in prompts:
            chunk_summary, chunk_time = generate_chunk_summary(chunk_prompt, max_new_tokens=100)
            chunk_summaries.append(chunk_summary)
            total_time += chunk_time

        # Combine and summarize
        final_prompt = build_final_prompt(chunk_summaries)
        
        final_prompt_ids = tokenizer(final_prompt, truncation=False, return_tensors="pt")["input_ids"][0]
        if final_prompt_ids.shape[0] > MAX_CONTEXT - MAX_NEW_TOKENS:
            allowed_tokens = MAX_CONTEXT - MAX_NEW_TOKENS
            # Keeps the last N tokens. 
            final_prompt = tokenizer.decode(final_prompt_ids[-allowed_tokens:], skip_special_tokens=True)

        gen_text, final_time = generate_response(final_prompt)
        total_time += final_time

    generated_texts.append(gen_text)
    generation_times.append(total_time)  # Append the true cumulative time
    
    if (i + 1) % 10 == 0 or (i + 1) == len(test_df):
        print(f"Generated {i + 1}/{len(test_df)} responses")

test_df = test_df.copy()
test_df["generated_text"] = generated_texts
test_df["generation_time_sec"] = generation_times


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Generated 10/100 responses
Generated 20/100 responses
Generated 30/100 responses
Generated 40/100 responses
Generated 50/100 responses
Generated 60/100 responses
Generated 70/100 responses
Generated 80/100 responses
Generated 90/100 responses
Generated 100/100 responses


## 6. Extract the Summary Section from Generated Output

Since the prompt is seeded with `Summary:`, the generated continuation
starts as the summary and (ideally) continues into `Key Points:`,
`Vocabulary:`, and `Quiz:`. The **Summary** portion is extracted for a
fair, like-for-like comparison against the single-sentence gold XSum
summary in the ROUGE and compression-ratio metrics.


In [8]:
SECTION_HEADERS = ["Key Points:", "Vocabulary:", "Quiz:"]

def extract_summary_section(generated_text: str) -> str:
    text = generated_text.strip()
    cut_positions = [text.find(h) for h in SECTION_HEADERS if text.find(h) != -1]
    if cut_positions:
        text = text[: min(cut_positions)]
    return text.strip()


test_df["generated_summary"] = test_df["generated_text"].apply(extract_summary_section)
test_df[["article", "summary", "generated_summary"]].head()


,article,summary,generated_summary
0,Dun Deardail was constructed on a prominent kn...,Archaeologists are preparing for the first eve...,A new excavation of a Scottish fort is to begi...
1,The Volkswagen scandal underlined the inadequa...,German scientists have developed a smart way t...,The Heidelberg team is working on a new system...
2,"Both sides have overlapping claims in the sea,...",The Philippines has submitted evidence to a UN...,The Philippines has accused China of violating...
3,"In just two words, Josh Hawkins teaches the ba...",A youth pastor from Sydney has created a stir ...,Australian actor Josh Hawkins has spoken out a...
4,"A search for the ""very large animal"" seen in S...","A woman has claimed sightings of a ""lion""- whi...",A cat that was spotted in a field in Essex has...


## 7. Evaluate: ROUGE-1, ROUGE-2, ROUGE-L, Flesch Reading Ease, Compression Ratio, Generation Time

- **ROUGE-1 / ROUGE-2 / ROUGE-L** — overlap between the generated summary
  and the gold XSum summary (F1, via `rouge_score`, with stemming).
- **Flesch Reading Ease** — readability of the full generated response
  (higher = easier to read); measures how "simplified" the output is.
- **Compression Ratio** — `len(generated summary) / len(original article)`
  in words; lower means the model compressed the article more.
- **Generation Time** — wall-clock seconds to generate each response.


In [9]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def compute_rouge(reference: str, hypothesis: str):
    scores = scorer.score(reference, hypothesis)
    return {
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
    }


def word_count(text: str) -> int:
    return max(1, len(text.split()))


rouge1_scores, rouge2_scores, rougeL_scores = [], [], []
flesch_scores = []
compression_ratios = []

for _, row in test_df.iterrows():
    rouge = compute_rouge(row["summary"], row["generated_summary"])
    rouge1_scores.append(rouge["rouge1"])
    rouge2_scores.append(rouge["rouge2"])
    rougeL_scores.append(rouge["rougeL"])

    # Flesch Reading Ease on the full generated response
    flesch_scores.append(textstat.flesch_reading_ease(row["generated_text"]) if row["generated_text"] else 0.0)

    # Compression ratio: generated summary length vs. original article length
    compression_ratios.append(word_count(row["generated_summary"]) / word_count(row["article"]))

test_df["rouge1"] = rouge1_scores
test_df["rouge2"] = rouge2_scores
test_df["rougeL"] = rougeL_scores
test_df["flesch_reading_ease"] = flesch_scores
test_df["compression_ratio"] = compression_ratios


## 8. Aggregate Results and Save

In [10]:
summary_metrics = pd.DataFrame({
    "metric": [
        "ROUGE-1", "ROUGE-2", "ROUGE-L",
        "Flesch Reading Ease", "Compression Ratio", "Generation Time (sec)",
    ],
    "mean": [
        test_df["rouge1"].mean(),
        test_df["rouge2"].mean(),
        test_df["rougeL"].mean(),
        test_df["flesch_reading_ease"].mean(),
        test_df["compression_ratio"].mean(),
        test_df["generation_time_sec"].mean(),
    ],
    "std": [
        test_df["rouge1"].std(),
        test_df["rouge2"].std(),
        test_df["rougeL"].std(),
        test_df["flesch_reading_ease"].std(),
        test_df["compression_ratio"].std(),
        test_df["generation_time_sec"].std(),
    ],
})

print(summary_metrics.to_string(index=False))

# Save per-example results and the aggregate summary
per_example_path = os.path.join(RESULTS_DIR, "opt125m_test_results.csv")
summary_path = os.path.join(RESULTS_DIR, "opt125m_evaluation_summary.csv")

test_df.drop(columns=["input_ids", "chunk", "chunk_count", "prompt", "prompt_word_count", "prompt_token_length"], errors="ignore").to_csv(per_example_path, index=False)
summary_metrics.to_csv(summary_path, index=False)

print(f"\nSaved per-example results to '{per_example_path}'")
print(f"Saved evaluation summary to '{summary_path}'")


               metric      mean       std
              ROUGE-1  0.294436  0.132380
              ROUGE-2  0.077952  0.098123
              ROUGE-L  0.227913  0.116860
  Flesch Reading Ease 61.535530 20.481757
    Compression Ratio  0.111549  0.109190
Generation Time (sec)  0.222423  0.158670


PermissionError: [Errno 13] Permission denied: 'results\\opt125m_test_results.csv'